# FULL — Utah FORGE 56-32 Geothermal Drilling Sensor Clustering with K-Means (Kaggle, English)

**Title (IEEE):** Impact of Feature Scaling and Temporal Window Aggregation on Multivariate Geothermal Drilling Sensor Clustering Using K-Means

**One-notebook final version.** Merges `01_eda` + `02_scaling` + `03_window` + `04_clustering` into a single Kaggle run (~20–30 min on CPU, RAM-safe).

## Step-by-step map (use this table in paper §II as pipeline figure)

| Step | Goal | Input → Output | Paper section |
|---|---|---|---|
| 0. Setup | Resolve Kaggle input path, set seed=42 | Dataset `utah-forge-56-32-1sec` → `df` | Reproducibility note |
| 1. EDA | Descriptive stats, scale audit, correlation, boxplot, `-999.25` audit | 80k sample → `table1_descriptive.csv`, `fig1_correlation.png`, `fig2_boxplots.png` | §II-A Data Understanding |
| 2. Scaling | Prove Euclidean bias; compare Standard vs Robust | 20k sample, k=4 demo → `table2_scaler_compare.csv`, `fig3_scaling_hist.png` | §II-B Preprocessing |
| 3. Windowing | Keep temporal dependency (Keogh & Lin 2005: cluster features, not raw subsequences) | 30k rows → 60s/30s mean+std+delta (24-D) → `table3_temporal_compare.csv`, `fig4_temporal_trace.png` | §II-C Temporal Aggregation |
| 4. Clustering | Elbow k=2..8 × 4 scenarios, Silhouette + DBI, centroids → Coley (2019) regimes | 60k sample → `table4_elbow_metrics.csv`, `fig5_elbow_silhouette.png`, `table5_centroids.csv`, `fig6_pca_clusters.png` | §III Results |
| 5. Export | Download `/kaggle/output/` for IEEE tables/figures | All tables+figures | Appendix + repo link |

> **Kaggle setup:** New Notebook → Add Input → private dataset `utah-forge-56-32-1sec` (the 357 MB CSV). Runtime: CPU, no GPU needed. All captions/labels are English-ready.

#### Cell: Setup — imports, constants, input resolver
**What:** imports libraries, defines the 8-sensor list (`SENSOR_8`), sample sizes, `SEED=42`, and `resolve_input()` which finds the CSV inside `/kaggle/input` (recursive + fuzzy match) with a local `dataset/` fallback.
**Why:** one seed and one path rule keep every run reproducible on Kaggle and locally.
**Read this:** the printed `INPUT` path (must point to your CSV) and `OUTPUT` folder (`/kaggle/output` on Kaggle).
**Folders:** this notebook never sees your laptop's `datmin/` folder. On Kaggle it runs on Kaggle's server: dataset comes from `/kaggle/input` (Add Input), outputs go to `/kaggle/working` (download them back into `datmin/figures/` afterwards). Locally the same code falls back to `dataset/` and `../figures/`.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

SENSOR_8 = ['Rate Of Penetration (ft_per_hr)','Weight on Bit (klbs)','Rotary RPM (RPM)','Standpipe Pressure (psi)','Rotary Torque (kft_lb)','Hook Load (klbs)','Total Pump Output (gal_per_min)','Differential Pressure (psi)']
CSV_NAME = '56-32 1sec data 27029986.csv'
SEED = 42
N_EDA, N_SCALE, N_WIN_SRC, N_FINAL = 80000, 20000, 30000, 60000

def resolve_input(csv=CSV_NAME):
    root = Path('/kaggle/input')
    if root.exists():
        hits = list(root.rglob(csv))
        if hits:
            return hits[0]
        for p in root.rglob('*.csv'):
            if '56-32' in p.name or '1sec' in p.name or '27029986' in p.name:
                return p
        csvs = [p for p in root.rglob('*.csv') if p.is_file()]
        if csvs:
            return max(csvs, key=lambda p: p.stat().st_size)
        pqs = [p for p in root.rglob('*.parquet') if p.is_file()]
        if pqs:
            return max(pqs, key=lambda p: p.stat().st_size)
    for c in [Path('dataset')/csv, Path('../dataset')/csv, Path.cwd()/'dataset'/csv]:
        if c.exists(): return c
    listing = [str(p) for p in sorted(root.rglob('*'))][:30] if root.exists() else ['</kaggle/input not found>']
    raise FileNotFoundError('Dataset CSV not found. /kaggle/input contains: ' + str(listing) + ' | Fix: right panel Add Input -> add your private dataset -> re-run.')

if Path('/kaggle/working').exists(): OUT = Path('/kaggle/working')  # live session: visible in file browser, downloadable, saved on Save Version
elif Path('/kaggle/output').exists(): OUT = Path('/kaggle/output')
else: OUT = Path('../figures')
OUT.mkdir(parents=True, exist_ok=True)
print('INPUT:', resolve_input())
print('OUTPUT:', OUT)

## Step 1 — EDA (Data Understanding)
**Why:** (a) quantify scale disparity (psi ~thousands vs torque ~tens → Euclidean bias), (b) confirm `Pason Gas`/`Gamma` are inactive (`-999.25`), (c) correlation + outliers for sensor selection justification (need ≥5 continuous numeric, unlabeled).

#### Cell: Load EDA sample
**What:** reads only the needed columns (`usecols`), downcasts to `float32`, takes a random 80k-row sample, and builds a `datetime` index from date + time columns.
**Why:** the full 2.5M-row file is too heavy for EDA; sampling + `float32` keeps RAM safe while staying representative.
**Read this:** printed shape and time range — confirms the sample loaded correctly.


In [ ]:
path = resolve_input()
usecols = ['YYYY/MM/DD','HH:MM:SS'] + SENSOR_8
eda = pd.read_csv(path, usecols=usecols, nrows=N_EDA*2, low_memory=True)
rng = np.random.RandomState(SEED)
if len(eda) > N_EDA:
    eda = eda.iloc[np.sort(rng.choice(len(eda), N_EDA, replace=False))].reset_index(drop=True)
for c in SENSOR_8: eda[c] = pd.to_numeric(eda[c], errors='coerce').astype('float32')
eda['datetime'] = pd.to_datetime(eda['YYYY/MM/DD']+' '+eda['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
print(eda.shape, eda['datetime'].min(), '->', eda['datetime'].max())
display(eda.head(2))

#### Cell: Descriptive statistics → `table1_descriptive.csv`
**What:** computes mean, std, min/max, quartiles, skewness, and range per sensor and saves Table 1.
**Why:** quantifies the scale disparity (psi in thousands vs torque in tens) — the core problem statement for the paper.
**Read this:** compare the `std`/`range` of Standpipe Pressure against Rotary Torque and WOB; that gap is the Euclidean-bias evidence.


In [ ]:
desc = eda[SENSOR_8].describe(percentiles=[.25,.5,.75]).T
desc['skew'] = eda[SENSOR_8].skew(numeric_only=True)
desc['range'] = desc['max'] - desc['min']
display(desc.round(2))
desc.round(2).to_csv(OUT/'table1_descriptive.csv')
# Reading guide: compare std/range of Standpipe Pressure vs Rotary Torque — that gap IS the scale-disparity problem.

#### Cell: Missing-code audit (`-999.25`)
**What:** checks the fraction of `-999.25` sentinel values in `Pason Gas` and `Gamma` on a 20k-row probe.
**Why:** `-999.25` means an inactive sensor, not a real measurement; both columns are ~100% sentinel so they must be dropped.
**Read this:** both ratios ≈ 1.0 → decision: DROP both, keep the 8-sensor set.


In [ ]:
chk = pd.read_csv(path, usecols=['Pason Gas (percent)','Gamma (api)'], nrows=20000)
print('Pason Gas == -999.25:', float((chk['Pason Gas (percent)']==-999.25).mean()))
print('Gamma == -999.25:', float((chk['Gamma (api)']==-999.25).mean()))
print('Decision: DROP both (inactive sensors). Keep the 8-sensor set.')

#### Cell: Correlation + boxplots → `fig1`, `fig2`
**What:** draws the 8×8 correlation heatmap and one boxplot per sensor (clipped to 1st–99th percentile for readability).
**Why:** correlation reveals redundant/coupled sensors; boxplots reveal outliers before clustering.
**Read this:** strong off-diagonal correlations (pump ↔ pressure) and long boxplot whiskers = real drilling outliers to discuss, not delete blindly.


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(eda[SENSOR_8].corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Fig 1. Correlation Matrix — 8 Geothermal Drilling Sensors')
plt.tight_layout(); plt.savefig(OUT/'fig1_correlation.png', dpi=150); plt.show()

fig, ax = plt.subplots(2,4, figsize=(14,6))
for a, c in zip(ax.ravel(), SENSOR_8):
    v = eda[c].dropna().to_numpy()
    lo, hi = np.percentile(v,[1,99])
    a.boxplot(v[(v>=lo)&(v<=hi)], vert=True); a.set_title(c.split(' (')[0], fontsize=9)
plt.suptitle('Fig 2. Boxplots (1st–99th pct clipped) — Outlier Detection')
plt.tight_layout(); plt.savefig(OUT/'fig2_boxplots.png', dpi=150); plt.show()

## Step 2 — Scaling (StandardScaler vs RobustScaler)
**Why:** K-Means uses Euclidean distance. Without scaling, `Standpipe Pressure` (0–3500 psi) dominates `Torque` (1–25) and `WOB` (0–50). `RobustScaler` (median/IQR) should resist drilling outliers better than `StandardScaler` (mean/std). Demo with k=4; full comparison repeats in Step 4.

#### Cell: Scaler comparison → `table2_scaler_compare.csv`
**What:** prints raw std per sensor, then runs K-Means (k=4) three ways — unscaled, `StandardScaler`, `RobustScaler` — reporting Silhouette and Davies-Bouldin.
**Why:** proves the scale problem empirically and picks the scaler for the final model; Robust (median/IQR) should resist drilling outliers better than Standard (mean/std).
**Read this:** higher Silhouette is better, lower DBI is better; unscaled should score worst.


In [ ]:
s = pd.read_csv(path, usecols=SENSOR_8, nrows=N_SCALE*2, low_memory=True)
for c in SENSOR_8: s[c]=pd.to_numeric(s[c], errors='coerce').astype('float32')
s = s.replace(-999.25, np.nan).dropna().reset_index(drop=True).sample(N_SCALE, random_state=SEED).to_numpy()
print(pd.DataFrame({'sensor':SENSOR_8,'std':s.std(axis=0)}).sort_values('std',ascending=False).to_string(index=False))
res2={}
for name, sc in [('unscaled',None),('standard',StandardScaler()),('robust',RobustScaler())]:
    X = s if sc is None else sc.fit_transform(s)
    lab = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(X)
    res2[name]={'silhouette':float(silhouette_score(X,lab)), 'dbi':float(davies_bouldin_score(X,lab))}
    print(name, {k:round(v,4) for k,v in res2[name].items()})
pd.DataFrame(res2).T.round(4).to_csv(OUT/'table2_scaler_compare.csv')
print('Higher silhouette = better; lower DBI = better.')

#### Cell: Distribution plot → `fig3_scaling_hist.png`
**What:** overlaid log-scale histograms of raw vs Standard-scaled vs Robust-scaled values.
**Why:** visual proof that scaling puts all sensors on a comparable footing before Euclidean distance is computed.
**Read this:** raw = one sharp spike + long tail; scaled versions should look compact and centered.


In [ ]:
xs, ss, rs = s, StandardScaler().fit_transform(s), RobustScaler().fit_transform(s)
fig, ax = plt.subplots(1,3, figsize=(12,3), sharey=True)
for a, X, t in zip(ax,[xs,ss,rs],['Raw','StandardScaler','RobustScaler']):
    a.hist(X.ravel(), bins=80); a.set_title(t); a.set_yscale('log')
plt.suptitle('Fig 3. Value Distribution: Raw vs Scaled (log-y)')
plt.tight_layout(); plt.savefig(OUT/'fig3_scaling_hist.png', dpi=150); plt.show()

## Step 3 — Temporal Windowing (bonus: temporal dependency)
**Why (Keogh & Lin 2005):** clustering raw subsequences gives meaningless sinusoidal centroids. Fix: aggregate each 60 s window (step 30 s) into `mean + std + delta` per sensor → 24-D feature vector that preserves dynamics (level, variability, trend).

#### Cell: Sliding-window features + point-vs-window → `table3_temporal_compare.csv`
**What:** sorts 30k rows by time, aggregates each 60 s window (step 30 s) into mean + std + delta per sensor (24-D vector), then compares K-Means Silhouette of point-based vs window-based representations.
**Why:** per Keogh & Lin (2005), clustering raw subsequences yields meaningless centroids — windowed statistical features preserve level, variability, and trend (the temporal-dependency bonus).
**Read this:** window silhouette should beat point silhouette; the saved table is the evidence.


In [ ]:
W, STEP = 60, 30
uw = pd.read_csv(path, usecols=['YYYY/MM/DD','HH:MM:SS']+SENSOR_8, nrows=N_WIN_SRC, low_memory=True)
for c in SENSOR_8: uw[c]=pd.to_numeric(uw[c], errors='coerce').astype('float32')
uw = uw.replace(-999.25, np.nan).dropna().reset_index(drop=True)
uw['datetime']=pd.to_datetime(uw['YYYY/MM/DD']+' '+uw['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
uw=uw.sort_values('datetime').reset_index(drop=True)
rows=[]
for st in range(0, len(uw)-W+1, STEP):
    ch=uw.iloc[st:st+W]; rec={'t0':ch['datetime'].iloc[0]}
    for c in SENSOR_8:
        v=ch[c].to_numpy(dtype='float64')
        rec[c+'__mean']=float(v.mean()); rec[c+'__std']=float(v.std()); rec[c+'__delta']=float(v[-1]-v[0])
    rows.append(rec)
win=pd.DataFrame(rows); feat=[c for c in win.columns if c!='t0']
print('windows:', win.shape, '| features:', len(feat))
Xp=RobustScaler().fit_transform(uw[SENSOR_8].sample(min(15000,len(uw)), random_state=SEED))
lp=KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(Xp)
Xw=RobustScaler().fit_transform(win[feat].to_numpy())
lw=KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(Xw)
print(f"point silhouette: {silhouette_score(Xp,lp):.4f} | window-60s silhouette: {silhouette_score(Xw,lw):.4f}")
pd.DataFrame({'representation':['point','window-60s'],'silhouette':[float(silhouette_score(Xp,lp)), float(silhouette_score(Xw,lw))]}).to_csv(OUT/'table3_temporal_compare.csv', index=False)

#### Cell: Temporal trace → `fig4_temporal_trace.png`
**What:** plots the first 3000 seconds of Standpipe Pressure over time.
**Why:** visual evidence that consecutive records are temporally dependent (the assignment's key challenge), motivating the windowing above.
**Read this:** smooth regimes and abrupt jumps = operating-state transitions the clusters should recover.


In [ ]:
plt.figure(figsize=(10,3))
plt.plot(uw['datetime'].iloc[:3000], uw['Standpipe Pressure (psi)'].iloc[:3000], lw=0.7)
plt.title('Fig 4. Standpipe Pressure (first 3000 s) — temporal dependency')
plt.xlabel('Time'); plt.ylabel('psi'); plt.tight_layout()
plt.savefig(OUT/'fig4_temporal_trace.png', dpi=150); plt.show()

## Step 4 — Final Clustering (Elbow + Silhouette + Davies-Bouldin)
**Design:** 4 scenarios = {point, window-60s} × {Standard, Robust}. Elbow k=2..8 on 20k subsample per scenario (fast). Then refit best (default `window60-robust`, k=4 → Drilling / Circulation / Connection-Tripping / Idle per Coley 2019) and map centroids back to physical units.

#### Cell: Final sample + 4 scenarios
**What:** draws the 60k-row modeling sample and builds the four experiment arms: {point, window-60s} × {Standard, Robust}.
**Why:** this is the controlled experiment of the paper — scaler effect and temporal effect measured separately.
**Read this:** printed shapes per scenario; window arms have 24 feature columns instead of 8.


In [ ]:
usecols=['YYYY/MM/DD','HH:MM:SS']+SENSOR_8
fd=pd.read_csv(path, usecols=usecols, nrows=N_FINAL*2, low_memory=True)
n_take = min(len(fd), N_FINAL)
if len(fd) > n_take:
    fd=fd.iloc[np.sort(np.random.RandomState(SEED).choice(len(fd), n_take, replace=False))].reset_index(drop=True)
for c in SENSOR_8: fd[c]=pd.to_numeric(fd[c], errors='coerce').astype('float32')
fd=fd.replace(-999.25, np.nan).dropna().reset_index(drop=True)
fd['datetime']=pd.to_datetime(fd['YYYY/MM/DD']+' '+fd['HH:MM:SS'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
print('final sample:', fd.shape)
def windowed(d, w=60, step=30):
    d=d.sort_values('datetime').reset_index(drop=True); out=[]
    for s_ in range(0, len(d)-w+1, step):
        ch=d.iloc[s_:s_+w]; rec={'t0':ch['datetime'].iloc[0]}
        for c in SENSOR_8:
            v=ch[c].to_numpy(dtype='float64')
            rec[c+'__mean']=v.mean(); rec[c+'__std']=v.std(); rec[c+'__delta']=v[-1]-v[0]
        out.append(rec)
    return pd.DataFrame(out)
scen={'point-standard':(fd[SENSOR_8].to_numpy(),StandardScaler()),'point-robust':(fd[SENSOR_8].to_numpy(),RobustScaler())}
wf=windowed(fd); feat2=[c for c in wf.columns if c!='t0']
scen['window60-standard']=(wf[feat2].to_numpy(),StandardScaler())
scen['window60-robust']=(wf[feat2].to_numpy(),RobustScaler())
print({k:v[0].shape for k,v in scen.items()})

#### Cell: Elbow loop k=2..8 → `table4_elbow_metrics.csv`
**What:** for each scenario and each k, fits K-Means on a 20k subsample and records inertia, Silhouette, and Davies-Bouldin.
**Why:** Elbow (inertia drop) plus internal metrics jointly select the optimal k without any labels.
**Read this:** the two pivot tables — look for the k where inertia bends and Silhouette peaks.


In [ ]:
K_RANGE=range(2,9); rows=[]
for name,(X,sc) in scen.items():
    Xs=sc.fit_transform(X)
    idx=np.random.RandomState(SEED).choice(len(Xs), min(len(Xs),20000), replace=False); Xe=Xs[idx]
    for k in K_RANGE:
        km=KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xe)
        lab=km.labels_
        rows.append({'scenario':name,'k':k,'inertia':float(km.inertia_),'silhouette':float(silhouette_score(Xe,lab)),'dbi':float(davies_bouldin_score(Xe,lab))})
res=pd.DataFrame(rows); res.to_csv(OUT/'table4_elbow_metrics.csv', index=False)
display(res.pivot_table(index='k', columns='scenario', values='silhouette').round(4))
display(res.pivot_table(index='k', columns='scenario', values='dbi').round(4))

#### Cell: Elbow plots → `fig5_elbow_silhouette.png`
**What:** line plots of inertia-vs-k and Silhouette-vs-k for all four scenarios.
**Why:** the paper's main model-selection figure; readers pick k visually from the elbow.
**Read this:** the bend (elbow) in the left panel and the peak in the right panel — update `BEST_K` in the next cell to match.


In [ ]:
fig, ax=plt.subplots(1,2, figsize=(12,4))
for name,g in res.groupby('scenario'):
    ax[0].plot(g['k'], g['inertia'], marker='o', label=name)
    ax[1].plot(g['k'], g['silhouette'], marker='o', label=name)
ax[0].set_title('Fig 5a. Elbow — Inertia vs k'); ax[0].set_xlabel('k')
ax[1].set_title('Fig 5b. Silhouette vs k (higher better)'); ax[1].set_xlabel('k')
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'fig5_elbow_silhouette.png', dpi=150); plt.show()

#### Cell: Best-model refit → `table5_centroids.csv`, `table6_cluster_share.csv`
**What:** refits K-Means with the chosen scenario/k, maps centroids back to physical units, and saves cluster sizes.
**Why:** centroids in psi, klbs, RPM are physically interpretable; map them with Coley (2019): WOB>0+RPM>0+ROP>0 = Active Drilling, WOB≈0+ROP=0+pump high = Circulation, pump≈0+RPM≈0 = Connection/Idle.
**Read this:** each centroid row should read like a real rig state — that mapping is your §III discussion.


In [ ]:
BEST_SCENARIO='window60-robust'; BEST_K=4  # change after reading Fig 5
Xb, scb = scen[BEST_SCENARIO]
Xbs=scb.fit_transform(Xb)
kmf=KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Xbs)
labf=kmf.labels_
cent=scb.inverse_transform(kmf.cluster_centers_)
if 'window' in BEST_SCENARIO:
    mc=[c for c in feat2 if c.endswith('__mean')]
    ctab=pd.DataFrame(cent[:,:len(mc)], columns=[c.replace('__mean','') for c in mc])
else:
    ctab=pd.DataFrame(cent, columns=SENSOR_8)
ctab.index.name='cluster'; display(ctab.round(2))
ctab.round(2).to_csv(OUT/'table5_centroids.csv')
pd.Series(labf).value_counts(normalize=True).round(4).to_csv(OUT/'table6_cluster_share.csv')
print('Map with Coley (2019): WOB>0+RPM>0+ROP>0 => Active Drilling; WOB~0+ROP=0+pump high => Circulation & Hole Cleaning; pump~0+RPM~0 => Pipe Connection; else Idle.')

#### Cell: PCA scatter → `fig6_pca_clusters.png`
**What:** projects 15k points to 2-D with PCA and colors them by cluster.
**Why:** visual sanity check of cluster separation for the paper; compact, separated blobs = good clustering.
**Read this:** overlapping clouds suggest trying a different k or the window representation.


In [ ]:
pca_idx=np.random.RandomState(SEED).choice(len(Xbs), min(len(Xbs),15000), replace=False)  # one shared index: points and labels stay aligned
p2=PCA(n_components=2, random_state=SEED).fit_transform(Xbs[pca_idx])
ls=labf[pca_idx]
plt.figure(figsize=(6,5)); plt.scatter(p2[:,0], p2[:,1], c=ls, s=4, cmap='tab10', alpha=0.6)
plt.title(f'Fig 6. PCA 2D — {BEST_SCENARIO} k={BEST_K}')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.tight_layout()
plt.savefig(OUT/'fig6_pca_clusters.png', dpi=150); plt.show()

## Step 5 — Export checklist for IEEE paper
- Download `/kaggle/output/` → `table1–table6`, `fig1–fig6`.
- Tables → §II/§III; Figures → same numbers (already English captions).
- Appendix: Kaggle Dataset link + this Notebook link + GitHub repo link.
- Add Gen-AI declaration (required by assignment).
- If Elbow suggests k≠4, update `BEST_K` and re-run only the last 2 cells.

#### Cell: Export & verify — collect everything for download
**What:** lists what was actually written to `OUT`, copies it into `/kaggle/working` (the live, downloadable folder), and packs it into one `kaggle_outputs.zip`.
**Why:** if this cell reports `<EMPTY>`, the cells above never ran (error or skipped) — outputs cannot exist. The zip gives you one-click download instead of per-file clicks.
**Read this:** the printed file list (expect `table*.csv` + `fig*.png`) and the zip size; then download via the file browser.


In [ ]:
from pathlib import Path
import shutil, zipfile
OUT.mkdir(parents=True, exist_ok=True)
live = Path('/kaggle/working') if Path('/kaggle/working').exists() else OUT
moved = []
for f in sorted(OUT.glob('*')):
    if not f.is_file() or f.suffix == '.zip':
        continue
    t = live / f.name
    if f.resolve() != t.resolve():
        shutil.copy2(f, t)
    moved.append(f.name)
print('OUT =', OUT)
print('files:', moved if moved else '<EMPTY — cells above did not run or errored, re-run from the top>')
zp = live / 'kaggle_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in moved:
        z.write(live / name, name)
print(f'zip: {zp} ({zp.stat().st_size/1024:.1f} KB)')
print('Download: file browser ->', live, '-> kaggle_outputs.zip. For the paper link: Save Version -> Output tab.')

try:
    from IPython.display import FileLink, display as _display
    _display(FileLink(str(zp)))
    print('Click the link above to download the zip directly (no file browser needed).')
except Exception as _e:
    print('direct-link unavailable:', _e)
